# Fama-French 3-Factor Model 기반 Security Selection (2000-01 ~ 2020-12)

입력: `IFE_term_stock_data.csv` (CRSP 월별 패널) + Ken French 공식 3팩터(`F-F_Research_Data_Factors_CSV.zip` 해제본을 `data/`에 배치).

가정: ADJ_PRC 는 배당 재투자 반영 조정가격(총수익률). Colab에서는 `!pip install pandas_datareader` 후 첫 셀의 Drive 마운트 주석 해제·`BASE` 수정. 웹 다운로드 실패 시 로컬 파일을 자동으로 읽습니다.

In [ ]:
# =========================================================================
# Fama-French (1993) 3-Factor Model 기반 Security Selection
# =========================================================================
# 입력:
#   - IFE_term_stock_data.csv : CRSP 월별 패널 (PERMNO, date, EXCHCD, TICKER,
#                                COMNAM, ADJ_PRC, VOL), 2000-01 ~ 2020-12
#   - FF3 팩터 (Mkt-RF, SMB, HML, RF) :
#       1순위  pandas_datareader 로 Ken French 라이브러리에서 다운로드
#       2순위  로컬 파일 F-F_Research_Data_Factors.csv
#              (French 사이트 F-F_Research_Data_Factors_CSV.zip 해제 원본)
#   - 분석기간: 2000-01 ~ 2020-12 (SAMPLE_START / SAMPLE_END)
#   - 가정: ADJ_PRC 는 배당 재투자 반영 조정가격 → 가격비율 = 총수익률
#
# 수행:
#   [1] 데이터 로드 및 체크리스트 (중복, 결측, 연속월, 거래소)
#   [2] ADJ_PRC 기반 월별 수익률 산출 (연속월만) + 이상치 필터
#   [3] FF3 팩터 로드, % → 소수 변환, 초과수익률 산출
#   [4] 종목별 시계열 회귀:  R_i - RF = α_i + b_i(Mkt-RF) + s_i SMB + h_i HML + e_i
#       - 전체 표본(2000-01~) 과 최근 60개월 윈도우 두 가지
#   [5] Security selection 규칙 (in-sample):
#       - 관측치 ≥ MIN_OBS, α > 0, t(α) ≥ T_CUT, 유동성 하위 LIQ_CUT 제외
#       - 랭킹은 appraisal ratio (α / σ_resid), Treynor-Black 의 논리
#   [6] Out-of-sample 검증 (매년 12월 말 formation, 직전 60개월 회귀,
#       다음 12개월 동일가중 보유):
#       - 선정 포트폴리오 vs 시장, 사후 FF3 회귀로 OOS α 및 t(α)
#       - α 10분위 포트폴리오로 "과거 α 가 미래 α 를 예측하는가" 검정
#   [7] 결과 저장 및 요약 출력
#
# 산출물 (OUTPUT_DIR):
#   - ff3_regression_full_sample.csv     종목별 전체표본 회귀 결과
#   - ff3_regression_recent_window.csv   종목별 최근 60개월 회귀 결과
#   - ff3_selected_securities.csv        최종 선정 종목 리스트
#   - ff3_oos_portfolio_returns.csv      OOS 포트폴리오 월별 수익률
#   - ff3_oos_summary.csv                OOS 성과 요약
#   - ff3_alpha_decile_persistence.csv   α 분위별 사후 성과
#   - ff3_selection_summary.png          누적수익률 차트
# =========================================================================

import os
import sys
import warnings
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# -------------------------------------------------------------------------
# 0. 경로 및 파라미터
# -------------------------------------------------------------------------
# Colab 사용 시 아래 두 줄 주석 해제 후 BASE 를 Drive 경로로 변경
# from google.colab import drive; drive.mount('/content/drive')
# BASE = '/content/drive/MyDrive/하이핀 리서치 3조'

BASE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else "."
PATH_STOCK = os.path.join(BASE, "data", "IFE_term_stock_data.csv")
PATH_FF_LOCAL = os.path.join(BASE, "data", "F-F_Research_Data_Factors.csv")   # French 사이트 zip 해제 원본
OUTPUT_DIR = os.path.join(BASE, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAMPLE_START = pd.Period("2000-01", freq="M")   # 분석기간: 2000-01 ~ 2020-12
SAMPLE_END = pd.Period("2020-12", freq="M")     # (첫 수익률은 2000-02: 2000-01 가격이 기준가)

# [가정] ADJ_PRC 는 배당 재투자가 반영된 조정가격(총수익률 기준) → ADJ_PRC 비율 = 총수익률

MIN_OBS = 36          # 회귀 최소 관측치 (개월)
EST_WINDOW = 60       # 롤링/최근 윈도우 길이 (개월)
HOLD_MONTHS = 12      # OOS 보유 기간 (개월)
T_CUT = 2.0           # t(α) 하한
LIQ_CUT = 0.20        # formation 시점 직전 12개월 평균 VOL 하위 20% 제외
MAX_NAMES = 50        # 선정 종목 수 상한 (appraisal ratio 상위)
RET_CAP = 5.0         # 월수익률 +500% 초과는 데이터 오류로 간주하여 제거
N_DECILES = 10
VALID_EXCHCD = [1, 2, 3]   # NYSE / AMEX / NASDAQ


def print_section(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

In [ ]:
# -------------------------------------------------------------------------
# 1. 주가 데이터 로드 및 체크리스트
# -------------------------------------------------------------------------
print_section("[1] 주가 데이터 로드")
raw = pd.read_csv(PATH_STOCK, parse_dates=["date"])
print(f"행 수: {len(raw):,} | 종목(PERMNO) 수: {raw['PERMNO'].nunique():,} | "
      f"기간: {raw['date'].min().date()} ~ {raw['date'].max().date()}")

# [체크리스트] PERMNO-date 중복
dup = raw.duplicated(subset=["PERMNO", "date"]).sum()
print(f"[체크] PERMNO-date 중복: {dup:,}개")
if dup > 0:
    raw = raw.drop_duplicates(subset=["PERMNO", "date"], keep="first")

# [체크리스트] 핵심 결측
print(f"[체크] 결측치:\n{raw[['ADJ_PRC', 'VOL', 'EXCHCD']].isna().sum().to_string()}")
raw = raw.dropna(subset=["ADJ_PRC"])
raw = raw[raw["ADJ_PRC"] > 0]

# [체크리스트] 거래소 필터 (NYSE/AMEX/NASDAQ)
n_before = len(raw)
raw = raw[raw["EXCHCD"].isin(VALID_EXCHCD)].copy()
print(f"[체크] 거래소 필터(EXCHCD∈{VALID_EXCHCD}): {n_before:,} → {len(raw):,}")

raw["ym"] = raw["date"].dt.to_period("M")
raw = raw[(raw["ym"] >= SAMPLE_START) & (raw["ym"] <= SAMPLE_END)]
raw = raw.sort_values(["PERMNO", "ym"]).reset_index(drop=True)
print(f"[체크] 분석기간 필터 {SAMPLE_START} ~ {SAMPLE_END}: {len(raw):,}행")

In [ ]:
# -------------------------------------------------------------------------
# 2. 월별 수익률 산출 (연속월만) 및 이상치 필터
# -------------------------------------------------------------------------
print_section("[2] 월별 수익률 산출")
g = raw.groupby("PERMNO")
raw["prev_ym"] = g["ym"].shift(1)
raw["prev_prc"] = g["ADJ_PRC"].shift(1)
gap = (raw["ym"] - raw["prev_ym"]).apply(lambda x: x.n if pd.notna(x) else np.nan)

# 직전 관측이 정확히 1개월 전일 때만 수익률 계산 (월 누락 구간은 NaN)
raw["RET"] = np.where(gap == 1, raw["ADJ_PRC"] / raw["prev_prc"] - 1.0, np.nan)
print(f"[체크] 월 간격 분포 (1=연속): \n{gap.value_counts().head(4).to_string()}")

n_cap = (raw["RET"] > RET_CAP).sum()
raw.loc[raw["RET"] > RET_CAP, "RET"] = np.nan
print(f"[체크] RET > {RET_CAP:.0%} 제거: {n_cap}개")

panel = raw.dropna(subset=["RET"]).copy()
print("[RET 기초통계]")
print(panel["RET"].describe(percentiles=[0.01, 0.5, 0.99]).to_string())
print("※ 주의: DLRET(상장폐지 수익률)이 없어 상장폐지 직전 월까지의 수익률만 반영됨"
      " → 생존편의(survivorship) 방향의 상방 편의 가능")

In [ ]:
# -------------------------------------------------------------------------
# 3. FF3 팩터 로드
# -------------------------------------------------------------------------
print_section("[3] Fama-French 3 팩터 로드")


def load_ff3_from_web():
    """pandas_datareader 로 Ken French 라이브러리에서 월별 3팩터 다운로드."""
    import pandas_datareader.data as web
    ds = web.DataReader("F-F_Research_Data_Factors", "famafrench",
                        start="1990-01-01")
    ff = ds[0].copy()                       # index: PeriodIndex(M), 단위 %
    ff.index = ff.index.to_period("M") if not isinstance(ff.index, pd.PeriodIndex) else ff.index
    ff.index.name = "ym"
    return ff


def load_ff3_from_local(path):
    """French 사이트 원본 CSV 포맷 (헤더 3~4줄, 월별 블록 뒤 Annual 블록)."""
    lines = open(path, encoding="latin-1").read().splitlines()
    rows = []
    for ln in lines:
        parts = [p.strip() for p in ln.split(",")]
        if len(parts) == 5 and parts[0].isdigit() and len(parts[0]) == 6:
            rows.append(parts)
    ff = pd.DataFrame(rows, columns=["ym", "Mkt-RF", "SMB", "HML", "RF"])
    ff["ym"] = pd.PeriodIndex(ff["ym"].str[:4] + "-" + ff["ym"].str[4:], freq="M")
    ff = ff.set_index("ym").astype(float)
    return ff


ff_source = None
try:
    ff3 = load_ff3_from_web()
    ff_source = "Ken French library (pandas_datareader)"
except Exception as e:           # 인터넷 미연결 / 모듈 없음 → 로컬 파일
    print(f"⚠️ 웹 다운로드 실패 ({type(e).__name__}) → 로컬 파일 사용")
    ff3 = load_ff3_from_local(PATH_FF_LOCAL)
    ff_source = f"local file: {os.path.basename(PATH_FF_LOCAL)}"

ff3 = ff3[["Mkt-RF", "SMB", "HML", "RF"]].rename(columns={"Mkt-RF": "MKT_RF"})
ff3 = ff3 / 100.0                  # % → 소수
print(f"팩터 출처: {ff_source}")
print(f"팩터 원본 기간: {ff3.index.min()} ~ {ff3.index.max()} ({len(ff3)}개월)")
ff3 = ff3.loc[SAMPLE_START:SAMPLE_END]
print(f"분석기간 팩터: {ff3.index.min()} ~ {ff3.index.max()} ({len(ff3)}개월)")
print(ff3.describe().T[["mean", "std", "min", "max"]].to_string())

In [ ]:
# -------------------------------------------------------------------------
# 4. 병합 및 초과수익률
# -------------------------------------------------------------------------
print_section("[4] 병합 및 초과수익률")
data = panel.merge(ff3, left_on="ym", right_index=True, how="inner")
data["EXRET"] = data["RET"] - data["RF"]
print(f"병합 후 관측치: {len(data):,} | 기간: {data['ym'].min()} ~ {data['ym'].max()}")
lost = panel["ym"].max()
if lost > ff3.index.max():
    print(f"⚠️ 팩터 부재로 제외된 구간: {ff3.index.max() + 1} ~ {lost} "
          f"({(lost - ff3.index.max()).n}개월)")

# 월 단위 정렬 인덱스
months = np.sort(data["ym"].unique())
month_pos = {m: i for i, m in enumerate(months)}
data["t"] = data["ym"].map(month_pos)
F_all = ff3.reindex(months)[["MKT_RF", "SMB", "HML"]].to_numpy()

In [ ]:
# -------------------------------------------------------------------------
# 5. 종목별 FF3 회귀 (numpy 벡터화 OLS)
# -------------------------------------------------------------------------
def ols_ff3(y, X):
    """
    y: (n,) 초과수익률, X: (n,3) 팩터.  절편 포함 OLS.
    반환: alpha, b, s, h, t_alpha, t_b, t_s, t_h, r2, resid_std, nobs
    """
    n = len(y)
    Xc = np.column_stack([np.ones(n), X])
    k = Xc.shape[1]
    XtX_inv = np.linalg.inv(Xc.T @ Xc)
    beta = XtX_inv @ Xc.T @ y
    resid = y - Xc @ beta
    sse = resid @ resid
    dof = n - k
    sigma2 = sse / dof
    se = np.sqrt(np.diag(XtX_inv) * sigma2)
    tstat = beta / se
    sst = ((y - y.mean()) ** 2).sum()
    r2 = 1 - sse / sst if sst > 0 else np.nan
    return (*beta, *tstat, r2, np.sqrt(sigma2), n)


REG_COLS = ["alpha", "b_mkt", "s_smb", "h_hml",
            "t_alpha", "t_mkt", "t_smb", "t_hml", "r2", "resid_std", "nobs"]


def run_regressions(sub, label):
    """sub: EXRET, t, PERMNO 를 포함한 패널. 종목별로 MIN_OBS 이상이면 회귀."""
    out = []
    for permno, grp in sub.groupby("PERMNO", sort=False):
        if len(grp) < MIN_OBS:
            continue
        y = grp["EXRET"].to_numpy()
        X = F_all[grp["t"].to_numpy()]
        out.append((permno, *ols_ff3(y, X)))
    res = pd.DataFrame(out, columns=["PERMNO"] + REG_COLS)
    res["alpha_ann"] = res["alpha"] * 12                    # 연율화 α (단순 ×12)
    res["p_alpha"] = 2 * stats.t.sf(res["t_alpha"].abs(), df=res["nobs"] - 4)
    res["appraisal_ratio"] = res["alpha"] / res["resid_std"]   # 월 단위
    res["window"] = label
    return res


# 종목 식별 정보 (마지막 관측 기준)
ident = (raw.sort_values("ym").groupby("PERMNO")
         .agg(TICKER=("TICKER", "last"), COMNAM=("COMNAM", "last"),
              EXCHCD=("EXCHCD", "last"), last_ym=("ym", "last"),
              last_adj_prc=("ADJ_PRC", "last")))

print_section("[5-a] 전체 표본 회귀 (2000-01 ~ 팩터 종료월)")
reg_full = run_regressions(data, "full_sample")
print(f"회귀 종목 수: {len(reg_full):,} (관측치 ≥ {MIN_OBS})")
print("[α 분포 (월, 소수)]")
print(reg_full["alpha"].describe(percentiles=[0.05, 0.5, 0.95]).to_string())
print(f"t(α) ≥ {T_CUT}: {(reg_full['t_alpha'] >= T_CUT).sum():,}개 | "
      f"t(α) ≤ -{T_CUT}: {(reg_full['t_alpha'] <= -T_CUT).sum():,}개 | "
      f"평균 R²: {reg_full['r2'].mean():.3f}")

print_section(f"[5-b] 최근 {EST_WINDOW}개월 회귀")
last_m = months[-1]
recent_start = last_m - (EST_WINDOW - 1)
recent = data[data["ym"] >= recent_start]
reg_recent = run_regressions(recent, f"recent_{EST_WINDOW}m")
print(f"윈도우: {recent_start} ~ {last_m} | 회귀 종목 수: {len(reg_recent):,}")

In [ ]:
# -------------------------------------------------------------------------
# 6. Security selection (in-sample, 최근 윈도우 기준)
# -------------------------------------------------------------------------
print_section("[6] Security selection 규칙 적용")


def liquidity_filter(as_of_ym, lookback=12):
    """as_of 직전 lookback 개월 평균 VOL 하위 LIQ_CUT 를 제외한 PERMNO 집합."""
    w = raw[(raw["ym"] > as_of_ym - lookback) & (raw["ym"] <= as_of_ym)]
    avg_vol = w.groupby("PERMNO")["VOL"].mean()
    cut = avg_vol.quantile(LIQ_CUT)
    return set(avg_vol[avg_vol >= cut].index)


def select(res, as_of_ym, require_alive=True):
    liq = liquidity_filter(as_of_ym)
    cand = res.merge(ident, left_on="PERMNO", right_index=True, how="left")
    cond = (
        (cand["alpha"] > 0) &
        (cand["t_alpha"] >= T_CUT) &
        (cand["PERMNO"].isin(liq))
    )
    if require_alive:
        cond &= cand["last_ym"] >= as_of_ym
    sel = cand[cond].sort_values("appraisal_ratio", ascending=False).head(MAX_NAMES)
    return sel


selected = select(reg_recent, last_m)
selected = selected.reset_index(drop=True)
selected.index += 1
print(f"기준월: {last_m} | 후보(회귀) {len(reg_recent):,} → "
      f"조건 통과 {len(selected)}개 (α>0, t(α)≥{T_CUT}, 유동성 하위 {LIQ_CUT:.0%} 제외, "
      f"상한 {MAX_NAMES})")
show_cols = ["PERMNO", "TICKER", "COMNAM", "alpha", "alpha_ann", "t_alpha",
             "b_mkt", "s_smb", "h_hml", "r2", "appraisal_ratio", "nobs"]
pd.set_option("display.width", 200)
print(selected[show_cols].head(25).round(3).to_string())

# 전체표본 회귀에서도 통과하는 종목 (강건성)
sel_full = select(reg_full, last_m)
both = set(selected["PERMNO"]) & set(sel_full["PERMNO"])
selected["also_full_sample"] = selected["PERMNO"].isin(both)
print(f"\n전체표본 회귀에서도 조건 통과: {len(both)}개 / {len(selected)}개")

In [ ]:
# -------------------------------------------------------------------------
# 7. Out-of-sample 검증
# -------------------------------------------------------------------------
print_section("[7] Out-of-sample 검증 (매년 12월 formation, 직전 60개월 추정, 12개월 보유)")

# 월별 시장/무위험/팩터 (검증 기간용)
ff_m = ff3.reindex(months)
ret_wide = data.pivot(index="ym", columns="PERMNO", values="RET")   # 월×종목

formation_months = [m for m in months
                    if m.month == 12 and m >= months[0] + (MIN_OBS - 1)
                    and m + 1 <= months[-1]]

oos_rows = []          # 선정 포트폴리오 월별 수익률
decile_rows = []       # α 10분위 포트폴리오 월별 수익률
formation_log = []

for fm in formation_months:
    win = data[(data["ym"] > fm - EST_WINDOW) & (data["ym"] <= fm)]
    res = run_regressions(win, f"form_{fm}")
    if len(res) < 100:
        continue
    liq = liquidity_filter(fm)
    res = res[res["PERMNO"].isin(liq)]

    # (a) 선정 포트폴리오
    sel = res[(res["alpha"] > 0) & (res["t_alpha"] >= T_CUT)] \
        .sort_values("appraisal_ratio", ascending=False).head(MAX_NAMES)
    # (b) 대조군: t(α) ≤ -T_CUT (negative alpha)
    neg = res[(res["alpha"] < 0) & (res["t_alpha"] <= -T_CUT)] \
        .sort_values("appraisal_ratio").head(MAX_NAMES)
    # (c) α 10분위 (t(α) 기준으로 분위, 모든 회귀 가능 종목)
    res = res.copy()
    res["decile"] = pd.qcut(res["t_alpha"].rank(method="first"), N_DECILES,
                            labels=range(1, N_DECILES + 1)).astype(int)

    hold = [m for m in months if fm < m <= fm + HOLD_MONTHS]
    formation_log.append({"formation": str(fm), "n_regressed": len(res),
                          "n_selected": len(sel), "n_negative": len(neg)})

    for m in hold:
        r = ret_wide.loc[m]
        # 동일가중, 보유 중 상장폐지/결측 종목은 해당 월부터 제외 (남은 종목으로 재가중)
        r_sel = r.reindex(sel["PERMNO"]).dropna()
        r_neg = r.reindex(neg["PERMNO"]).dropna()
        r_all = r.reindex(res["PERMNO"]).dropna()
        oos_rows.append({
            "ym": m, "formation": str(fm),
            "ret_selected": r_sel.mean() if len(r_sel) else np.nan,
            "n_selected_alive": len(r_sel),
            "ret_negative": r_neg.mean() if len(r_neg) else np.nan,
            "ret_universe_ew": r_all.mean(),
            "MKT_RF": ff_m.loc[m, "MKT_RF"], "SMB": ff_m.loc[m, "SMB"],
            "HML": ff_m.loc[m, "HML"], "RF": ff_m.loc[m, "RF"],
        })
        for d, grp in res.groupby("decile"):
            rd = r.reindex(grp["PERMNO"]).dropna()
            decile_rows.append({"ym": m, "decile": d, "ret": rd.mean()})

oos = pd.DataFrame(oos_rows).set_index("ym")
oos["ex_selected"] = oos["ret_selected"] - oos["RF"]
oos["ex_negative"] = oos["ret_negative"] - oos["RF"]
oos["ex_universe"] = oos["ret_universe_ew"] - oos["RF"]
oos["ret_mkt"] = oos["MKT_RF"] + oos["RF"]
print(pd.DataFrame(formation_log).to_string(index=False))


def perf_summary(ex, name):
    """월별 초과수익률 시계열 → 연율화 통계 + 사후 FF3 회귀."""
    ex = ex.dropna()
    X = ff_m.loc[ex.index, ["MKT_RF", "SMB", "HML"]].to_numpy()
    a, b, s, h, ta, tb, ts, th, r2, rsd, n = ols_ff3(ex.to_numpy(), X)
    return {
        "portfolio": name, "months": n,
        "mean_ex_ret_ann": ex.mean() * 12, "vol_ann": ex.std() * np.sqrt(12),
        "sharpe": ex.mean() / ex.std() * np.sqrt(12),
        "ff3_alpha_ann": a * 12, "t_alpha": ta,
        "b_mkt": b, "s_smb": s, "h_hml": h, "r2": r2,
    }


summary = pd.DataFrame([
    perf_summary(oos["ex_selected"], f"Selected (t(α)≥{T_CUT}, top {MAX_NAMES})"),
    perf_summary(oos["ex_negative"], f"Negative-α control (t(α)≤-{T_CUT})"),
    perf_summary(oos["ex_universe"], "Universe EW"),
    perf_summary(oos["MKT_RF"], "Market (Mkt-RF)"),
])
# 시장 자기자신 회귀의 α, t(α) 는 정의상 0 → 표시 제외
summary.loc[summary["portfolio"].str.startswith("Market"), ["ff3_alpha_ann", "t_alpha"]] = np.nan
print("\n[OOS 성과 요약: 초과수익률 기준, 연율화]")
print(summary.round(4).to_string(index=False))

dec = pd.DataFrame(decile_rows)
dec_wide = dec.pivot(index="ym", columns="decile", values="ret")
dec_ex = dec_wide.sub(ff_m.loc[dec_wide.index, "RF"], axis=0)
dec_summary = pd.DataFrame([perf_summary(dec_ex[d], f"D{d}") for d in dec_ex.columns])
ls = dec_ex[N_DECILES] - dec_ex[1]
ls_stats = perf_summary(ls + 0, "D10 - D1")     # long-short: RF 가 상쇄되므로 그대로
print("\n[α 10분위 사후 성과 (D1 = 과거 t(α) 최저, D10 = 최고)]")
print(dec_summary[["portfolio", "mean_ex_ret_ann", "sharpe", "ff3_alpha_ann", "t_alpha"]]
      .round(4).to_string(index=False))
print(f"D10-D1 long-short: 연 {ls.mean()*12:.2%}, t = "
      f"{ls.mean()/ls.std()*np.sqrt(len(ls)):.2f}, FF3 α(연) {ls_stats['ff3_alpha_ann']:.2%} "
      f"(t={ls_stats['t_alpha']:.2f})")

# 형성연도별 hit-rate: 선정 포트폴리오가 시장을 이긴 해의 비율
by_year = oos.groupby("formation").agg(sel=("ret_selected", lambda x: (1 + x).prod() - 1),
                                      mkt=("ret_mkt", lambda x: (1 + x).prod() - 1),
                                      uni=("ret_universe_ew", lambda x: (1 + x).prod() - 1))
by_year["beat_mkt"] = by_year["sel"] > by_year["mkt"]
by_year["beat_universe"] = by_year["sel"] > by_year["uni"]
print("\n[형성연도별 다음 12개월 누적수익률]")
print(by_year.round(4).to_string())
print(f"시장 대비 승률: {by_year['beat_mkt'].mean():.0%} | "
      f"유니버스(EW) 대비 승률: {by_year['beat_universe'].mean():.0%}")

In [ ]:
# -------------------------------------------------------------------------
# 8. 저장 및 차트
# -------------------------------------------------------------------------
print_section("[8] 저장")
reg_full.merge(ident, left_on="PERMNO", right_index=True, how="left") \
    .to_csv(os.path.join(OUTPUT_DIR, "ff3_regression_full_sample.csv"), index=False)
reg_recent.merge(ident, left_on="PERMNO", right_index=True, how="left") \
    .to_csv(os.path.join(OUTPUT_DIR, "ff3_regression_recent_window.csv"), index=False)
selected.to_csv(os.path.join(OUTPUT_DIR, "ff3_selected_securities.csv"), index_label="rank")
oos.to_csv(os.path.join(OUTPUT_DIR, "ff3_oos_portfolio_returns.csv"))
summary.to_csv(os.path.join(OUTPUT_DIR, "ff3_oos_summary.csv"), index=False)
dec_summary.to_csv(os.path.join(OUTPUT_DIR, "ff3_alpha_decile_persistence.csv"), index=False)
by_year.to_csv(os.path.join(OUTPUT_DIR, "ff3_oos_by_formation_year.csv"))

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    cum = (1 + oos[["ret_selected", "ret_negative", "ret_universe_ew", "ret_mkt"]]).cumprod()
    cum.index = cum.index.to_timestamp()
    cum.rename(columns={"ret_selected": f"Selected (t(a)>={T_CUT})",
                        "ret_negative": "Negative-alpha control",
                        "ret_universe_ew": "Universe EW",
                        "ret_mkt": "Market"}).plot(ax=axes[0], logy=True)
    axes[0].set_title("OOS cumulative return (formation each Dec, 60m est., 12m hold)")
    axes[0].set_ylabel("Growth of 1 (log)")
    axes[0].grid(alpha=0.3)

    axes[1].bar(dec_summary["portfolio"], dec_summary["ff3_alpha_ann"] * 100)
    axes[1].set_title("Post-formation FF3 alpha by prior t(alpha) decile (%/yr)")
    axes[1].axhline(0, color="k", lw=0.8)
    axes[1].grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "ff3_selection_summary.png"), dpi=130)
    print("차트 저장 완료")
except Exception as e:
    print(f"차트 생략: {e}")

print(f"✔️ 모든 산출물 저장: {OUTPUT_DIR}")